# Job Market Demand Forecasting Using LSTM

## Deep Learning Project — Time Series Forecasting

**Objective:** Forecast monthly job posting volume per occupational sector using an LSTM neural network trained on the Indeed Job Postings Index (US market).

**Dataset:** [Indeed Job Postings Index](https://www.kaggle.com/datasets/kimminh21/job-postings) — Daily seasonally-adjusted job postings index across 20+ occupational sectors, baseline = 100 on Feb 1, 2020.

**Why LSTM?** Unlike classical models (ARIMA, Prophet), LSTMs maintain hidden state across time steps, learning non-linear dependencies such as COVID-era shock-recovery patterns, regime changes, and multi-step seasonal cycles that linear models cannot capture.

---

### Notebook Structure
1. Environment Setup & Imports
2. Dataset Loading & Exploration
3. Initial Exploratory Data Analysis (EDA)
4. Monthly Aggregation Pipeline
5. Data Normalization & Sequence Generation
6. LSTM Model Architecture & Training
7. Evaluation (RMSE, MAE)
8. Future Forecasting & Visualization
9. Interpretation, Limitations & Improvements

## 1. Environment Setup & Imports

All libraries used are **pre-installed on Kaggle**. No `pip install` needed.

| Library | Purpose |
|---------|---------|
| `pandas` | Data manipulation, time-series resampling |
| `numpy` | Numerical operations |
| `matplotlib` / `seaborn` | Static visualizations (report-quality) |
| `sklearn.preprocessing` | MinMaxScaler for normalization |
| `tensorflow.keras` | LSTM model building and training |

> **Kaggle Tip:** This notebook runs on **CPU**. LSTM training on our small dataset (~60 monthly points × 8 sectors) completes in under 2 minutes on CPU. No GPU needed.

In [3]:
# ============================================================
# 1. IMPORTS & CONFIGURATION
# ============================================================

import os
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'  # Suppress CUDA/TF registration warnings

import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Suppress non-critical warnings for cleaner output
warnings.filterwarnings('ignore')

# Plot styling — academic-quality figures
plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams.update({
    'figure.figsize': (14, 6),
    'figure.dpi': 100,
    'axes.titlesize': 14,
    'axes.labelsize': 12,
    'lines.linewidth': 2,
    'font.size': 11
})

# Reproducibility — fix all random seeds
SEED = 42
np.random.seed(SEED)

# TensorFlow import (Kaggle has TF pre-installed)
import tensorflow as tf
tf.random.set_seed(SEED)
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error

print(f"TensorFlow version: {tf.__version__}")
print(f"NumPy version:      {np.__version__}")
print(f"Pandas version:     {pd.__version__}")
print("✓ All imports successful")

TensorFlow version: 2.19.0
NumPy version:      2.0.2
Pandas version:     2.3.3
✓ All imports successful


## 2. Dataset Loading & Initial Exploration

### How to add the dataset on Kaggle
1. Open your Kaggle notebook
2. Click **"Add Data"** (right sidebar) → Search **"Indeed Job Postings Index"** by Kim Minh
3. Click **"Add"** — the dataset will be mounted at `/kaggle/input/job-postings/`

### Dataset structure
The Indeed dataset organizes data by country in separate folders. We use:
- `US/job_postings_by_sector_us.csv` — **daily index per occupational sector** (our main input)
- `US/aggregate_job_postings_us.csv` — total national postings (for context)
- `sector-job-title-examples.csv` — maps sector names to example job titles

### What to expect
- Each row = one day, one sector
- The `indeed_job_postings_index` column is already **seasonally adjusted** with baseline 100 = Feb 1, 2020
- Values > 100 mean more postings than pre-pandemic; values < 100 mean fewer

> **Debug tip:** If you get `FileNotFoundError`, check the exact folder name using the file listing cell below. Kaggle sometimes adds version suffixes to dataset folder names.

In [7]:
# ============================================================
# 2a. List all files in the dataset (verify correct mount path)
# ============================================================

INPUT_DIR = '/kaggle/input/datasets/kimminh21/job-postings'

print("Files in dataset:")
print("=" * 60)
for dirname, _, filenames in os.walk(INPUT_DIR):
    level = dirname.replace(INPUT_DIR, '').count(os.sep)
    indent = ' ' * 2 * level
    print(f"{indent}{os.path.basename(dirname)}/")
    subindent = ' ' * 2 * (level + 1)
    for f in sorted(filenames):
        filepath = os.path.join(dirname, f)
        size_mb = os.path.getsize(filepath) / (1024 * 1024)
        print(f"{subindent}{f} ({size_mb:.2f} MB)")

Files in dataset:
job-postings/
  .gitignore (0.00 MB)
  LICENSE (0.02 MB)
  README.md (0.01 MB)
  sector-job-title-examples.csv (0.00 MB)
  IT/
    aggregate_job_postings_IT.csv (0.18 MB)
  IE/
    aggregate_job_postings_IE.csv (0.18 MB)
  EA/
    aggregate_job_postings_EA.csv (0.18 MB)
  ES/
    aggregate_job_postings_ES.csv (0.18 MB)
  NL/
    aggregate_job_postings_NL.csv (0.18 MB)
  GB/
    aggregate_job_postings_GB.csv (0.18 MB)
    city_postings_gb.csv (3.62 MB)
    job_postings_by_sector_GB.csv (10.49 MB)
    regional_gb.csv (0.78 MB)
  US/
    aggregate_job_postings_US.csv (0.18 MB)
    job_postings_by_sector_US.csv (8.67 MB)
    metro_job_postings_us.csv (55.74 MB)
    state_job_postings_us.csv (2.29 MB)
  CA/
    aggregate_job_postings_CA.csv (0.18 MB)
    job_postings_by_sector_CA.csv (10.21 MB)
    metro_job_postings_CA.csv (3.29 MB)
    provincial_postings_ca.csv (0.45 MB)
  AU/
    aggregate_job_postings_AU.csv (0.18 MB)
    job_postings_by_sector_AU.csv (10.54 MB)
  FR/

In [ ]:
# ============================================================
# 2b. Load the US sector-level data (main dataset for LSTM)
# ============================================================

# Path to US sector data
SECTOR_FILE = os.path.join(INPUT_DIR, 'US', 'job_postings_by_sector_us.csv')

# Load with date parsing
df_raw = pd.read_csv(SECTOR_FILE, parse_dates=['date'])

print(f"Dataset shape: {df_raw.shape}")
print(f"Date range:    {df_raw['date'].min().date()} → {df_raw['date'].max().date()}")
print(f"Columns:       {list(df_raw.columns)}")
print(f"\nFirst 5 rows:")
df_raw.head()

In [ ]:
# ============================================================
# 2c. Basic dataset statistics
# ============================================================

print("Data types:")
print(df_raw.dtypes)
print(f"\nMissing values:\n{df_raw.isnull().sum()}")
print(f"\nUnique sectors: {df_raw['display_name'].nunique()}")
print(f"\nSector list:")
for i, sector in enumerate(sorted(df_raw['display_name'].unique()), 1):
    print(f"  {i:2d}. {sector}")

print(f"\nVariable types: {df_raw['variable'].unique()}")
print(f"\nIndex statistics:")
df_raw['indeed_job_postings_index'].describe()

In [ ]:
# ============================================================
# 2d. Filter: keep only 'total' postings (not 'new')
# ============================================================
# The dataset has two variables:
#   - 'total': all active job postings (what we want to forecast)
#   - 'new':   postings on Indeed for 7 days or fewer
# We use 'total' because it represents the full market demand signal.

df = df_raw[df_raw['variable'] == 'total'].copy()
df = df.drop(columns=['variable', 'jobcountry'])  # no longer needed

print(f"Filtered shape: {df.shape}")
print(f"Remaining columns: {list(df.columns)}")
df.head()

## 3. Exploratory Data Analysis (EDA)

Before building the LSTM, we need to understand the data's temporal structure:

1. **Overall trend**: How did job postings evolve from 2020 to 2026?
2. **COVID impact**: The pandemic caused a dramatic crash in Q1 2020 followed by an asymmetric recovery — this is the key non-linear pattern LSTM should capture.
3. **Sector differences**: Not all sectors recovered equally. Tech may have surged while hospitality lagged.
4. **Seasonality**: Are there recurring monthly patterns (e.g., hiring dips in December)?

### Sector Selection
We select **8 sectors** most relevant to data professionals and the tech job market. This keeps the scope manageable while providing enough variety for comparative analysis.

> **Expected output:** Line plots showing distinct trend shapes per sector, confirming the data has enough temporal complexity to justify LSTM over simpler methods.

In [ ]:
# ============================================================
# 3a. Select target sectors relevant to data/tech job market
# ============================================================
# We pick 8 sectors that are most relevant to our forecasting goal.
# These cover a range of recovery patterns for richer LSTM training.
#
# NOTE: Run cell 2c first to see all available sector names.
#       If a name doesn't match exactly, check the printed list and adjust.

TARGET_SECTORS = [
    'Software Development',
    'Data Science & Analysis',  
    'Information Technology',
    'Mathematics',
    'Marketing',
    'Management',
    'Banking & Finance',
    'Human Resources',
]

# Filter for target sectors
df_sectors = df[df['display_name'].isin(TARGET_SECTORS)].copy()

# Verify all sectors were found
found = df_sectors['display_name'].unique()
missing = set(TARGET_SECTORS) - set(found)
if missing:
    print(f"⚠️  Sectors NOT found in dataset: {missing}")
    print(f"   Available sectors: {sorted(df['display_name'].unique())}")
    print(f"   → Adjust TARGET_SECTORS names to match the dataset exactly.")
else:
    print(f"✓ All {len(TARGET_SECTORS)} sectors found")

print(f"\nFiltered dataset: {df_sectors.shape[0]:,} rows")
print(f"Date range: {df_sectors['date'].min().date()} → {df_sectors['date'].max().date()}")
print(f"Rows per sector: ~{df_sectors.shape[0] // len(found):,}")

In [ ]:
# ============================================================
# 3b. VISUALIZATION: Daily index trends per sector
# ============================================================
# This plot reveals:
# - The COVID crash (~March 2020)
# - Different recovery trajectories per sector
# - Whether trends have stabilized or are still shifting

fig, ax = plt.subplots(figsize=(16, 8))

for sector in TARGET_SECTORS:
    mask = df_sectors['display_name'] == sector
    sector_data = df_sectors[mask].sort_values('date')
    ax.plot(sector_data['date'], sector_data['indeed_job_postings_index'],
            label=sector, alpha=0.85)

ax.axhline(y=100, color='black', linestyle='--', alpha=0.4, label='Pre-pandemic baseline (100)')
ax.axvspan(pd.Timestamp('2020-03-01'), pd.Timestamp('2020-06-01'),
           alpha=0.1, color='red', label='COVID shock period')

ax.set_title('Indeed Job Postings Index — US Market (Daily)', fontsize=16)
ax.set_xlabel('Date')
ax.set_ylabel('Job Postings Index (100 = Feb 2020)')
ax.legend(loc='upper left', fontsize=9, ncol=2)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('daily_trends_all_sectors.png', dpi=150, bbox_inches='tight')
plt.show()
print("✓ Figure saved: daily_trends_all_sectors.png")

In [ ]:
# ============================================================
# 3c. VISUALIZATION: Distribution of index values per sector
# ============================================================
# Box plots reveal the spread and outliers for each sector.
# Sectors with wider distributions have more volatility — 
# these are harder to forecast but more interesting for LSTM.

fig, ax = plt.subplots(figsize=(14, 6))

sector_order = (df_sectors.groupby('display_name')['indeed_job_postings_index']
                .median().sort_values(ascending=False).index)

sns.boxplot(data=df_sectors, x='display_name', y='indeed_job_postings_index',
            order=sector_order, palette='viridis', ax=ax)

ax.axhline(y=100, color='red', linestyle='--', alpha=0.5, label='Baseline (100)')
ax.set_title('Distribution of Job Postings Index by Sector', fontsize=14)
ax.set_xlabel('Sector')
ax.set_ylabel('Index Value')
ax.set_xticklabels(ax.get_xticklabels(), rotation=35, ha='right')
ax.legend()

plt.tight_layout()
plt.savefig('sector_distributions.png', dpi=150, bbox_inches='tight')
plt.show()
print("✓ Figure saved: sector_distributions.png")

In [ ]:
# ============================================================
# 3d. VISUALIZATION: Missing data check (heatmap)
# ============================================================
# For time-series, gaps in data are critical. We check if any
# sector has missing days that could break our aggregation.

# Pivot to wide format: rows=dates, columns=sectors
df_pivot_check = df_sectors.pivot_table(
    index='date', columns='display_name',
    values='indeed_job_postings_index', aggfunc='first'
)

print(f"Pivoted shape: {df_pivot_check.shape}")
print(f"\nMissing values per sector:")
missing = df_pivot_check.isnull().sum()
print(missing)

# Visual: show data availability
fig, ax = plt.subplots(figsize=(14, 4))
sns.heatmap(df_pivot_check.isnull().T, cbar=False, cmap='Reds',
            xticklabels=False, ax=ax)
ax.set_title('Missing Data Heatmap (red = missing)', fontsize=14)
ax.set_xlabel('Date')
ax.set_ylabel('Sector')
plt.tight_layout()
plt.show()

total_missing = df_pivot_check.isnull().sum().sum()
total_cells = df_pivot_check.shape[0] * df_pivot_check.shape[1]
print(f"\nTotal missing: {total_missing}/{total_cells} ({100*total_missing/total_cells:.2f}%)")
if total_missing == 0:
    print("✓ No missing data — clean dataset!")
else:
    print("→ Missing values will be handled during monthly aggregation (mean ignores NaN by default)")

## 4. Monthly Aggregation Pipeline

### Why aggregate daily → monthly?

1. **Noise reduction**: Daily data contains high-frequency noise (weekday/weekend effects, holidays) that obscures the underlying demand signal. Monthly means smooth this out.
2. **Sequence length**: 5 years of daily data = ~1,800 points per sector. Monthly = ~60 points. For a university project LSTM, 60 points is the right scale — enough to learn patterns, small enough to train fast.
3. **Business relevance**: Hiring decisions are made on monthly/quarterly horizons, not daily. Monthly forecasts are more actionable.
4. **Aligns with project objective**: The project description specifies forecasting "for the next 3 to 6 **months**."

### Pipeline steps
1. Pivot daily data → wide format (dates × sectors)
2. Resample to month-end frequency using mean aggregation
3. Handle any remaining missing values (forward-fill)
4. Add cyclical time features (sin/cos month encoding)
5. Verify the result

> **Debug tip:** If the monthly DataFrame has unexpected NaN values, it usually means a sector started reporting later than others. Forward-fill or drop the first few months.

In [ ]:
# ============================================================
# 4a. Pivot to wide format and resample to monthly
# ============================================================

# Step 1: Pivot — rows=date, columns=sector_name, values=index
df_pivot = df_sectors.pivot_table(
    index='date',
    columns='display_name',
    values='indeed_job_postings_index',
    aggfunc='first'  # one value per date per sector
)

print(f"Daily pivot shape: {df_pivot.shape}")
print(f"Date range: {df_pivot.index.min().date()} → {df_pivot.index.max().date()}")

# Step 2: Resample daily → monthly (mean of daily values per month)
df_monthly = df_pivot.resample('ME').mean()

# Step 3: Handle any remaining NaN (forward-fill then backward-fill for edges)
nan_before = df_monthly.isnull().sum().sum()
df_monthly = df_monthly.ffill().bfill()
nan_after = df_monthly.isnull().sum().sum()

print(f"\nMonthly shape: {df_monthly.shape}")
print(f"Months: {len(df_monthly)} (from {df_monthly.index[0].strftime('%b %Y')} to {df_monthly.index[-1].strftime('%b %Y')})")
print(f"NaN filled: {nan_before} → {nan_after}")
print(f"\nFirst 5 months:")
df_monthly.head()

In [ ]:
# ============================================================
# 4b. VISUALIZATION: Monthly trends per sector (our LSTM input)
# ============================================================
# This is the actual data the LSTM will learn from.
# Compare with the daily plot above — monthly is much smoother.

fig, axes = plt.subplots(2, 4, figsize=(20, 10), sharex=True)
axes = axes.flatten()

for i, sector in enumerate(TARGET_SECTORS):
    ax = axes[i]
    ax.plot(df_monthly.index, df_monthly[sector], color='steelblue', linewidth=2)
    ax.axhline(y=100, color='red', linestyle='--', alpha=0.4)
    ax.set_title(sector, fontsize=11, fontweight='bold')
    ax.set_ylabel('Index')
    ax.grid(True, alpha=0.3)
    ax.tick_params(axis='x', rotation=45)

fig.suptitle('Monthly Job Postings Index per Sector (LSTM Input Data)', 
             fontsize=16, y=1.02)
plt.tight_layout()
plt.savefig('monthly_trends_per_sector.png', dpi=150, bbox_inches='tight')
plt.show()
print("✓ Figure saved: monthly_trends_per_sector.png")
print(f"→ Each subplot shows ~{len(df_monthly)} monthly data points — this is what the LSTM will train on.")

In [ ]:
# ============================================================
# 4c. VISUALIZATION: Correlation heatmap between sectors
# ============================================================
# High correlation means sectors move together (e.g., all tech sectors
# crashed and recovered together). This validates our choice to train
# separate univariate LSTMs — if correlation were perfect, a single
# model would suffice.

fig, ax = plt.subplots(figsize=(10, 8))

corr = df_monthly.corr()
mask = np.triu(np.ones_like(corr, dtype=bool))  # upper triangle mask

sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='coolwarm',
            center=0, square=True, linewidths=0.5, ax=ax,
            vmin=-1, vmax=1)

ax.set_title('Sector Correlation Matrix (Monthly Index)', fontsize=14)
plt.tight_layout()
plt.savefig('sector_correlation.png', dpi=150, bbox_inches='tight')
plt.show()
print("✓ Figure saved: sector_correlation.png")
print("→ High correlation (>0.8) = sectors move together")
print("→ Low correlation (<0.5) = independent trends — more interesting for separate forecasts")

In [ ]:
# ============================================================
# 4d. VISUALIZATION: Seasonality analysis
# ============================================================
# Group by month-of-year to see if there are recurring patterns.
# This helps justify adding sin/cos month features later.

df_monthly_copy = df_monthly.copy()
df_monthly_copy['month'] = df_monthly_copy.index.month
df_monthly_copy['year'] = df_monthly_copy.index.year

fig, axes = plt.subplots(2, 4, figsize=(20, 10))
axes = axes.flatten()

for i, sector in enumerate(TARGET_SECTORS):
    ax = axes[i]
    # Group by month across all years
    monthly_means = df_monthly_copy.groupby('month')[sector].agg(['mean', 'std'])
    
    ax.bar(monthly_means.index, monthly_means['mean'], 
           yerr=monthly_means['std'], capsize=3,
           color='steelblue', alpha=0.7, edgecolor='navy')
    ax.set_title(sector, fontsize=10, fontweight='bold')
    ax.set_xlabel('Month')
    ax.set_ylabel('Avg Index')
    ax.set_xticks(range(1, 13))
    ax.set_xticklabels(['J','F','M','A','M','J','J','A','S','O','N','D'])
    ax.grid(True, alpha=0.3, axis='y')

fig.suptitle('Seasonality Analysis: Average Index by Month (with std dev)', 
             fontsize=16, y=1.02)
plt.tight_layout()
plt.savefig('seasonality_analysis.png', dpi=150, bbox_inches='tight')
plt.show()
print("✓ Figure saved: seasonality_analysis.png")
print("→ If bars show consistent monthly patterns, sin/cos encoding will help the LSTM")
print("→ Large std bars = high year-to-year variance (COVID effect)")

# Clean up temp columns
df_monthly_copy.drop(columns=['month', 'year'], inplace=True)

In [ ]:
# ============================================================
# 4e. Summary statistics of the monthly dataset
# ============================================================
# This table can go directly into your report.

print("=" * 70)
print("MONTHLY DATASET SUMMARY — READY FOR LSTM")
print("=" * 70)
print(f"Shape: {df_monthly.shape[0]} months × {df_monthly.shape[1]} sectors")
print(f"Period: {df_monthly.index[0].strftime('%B %Y')} → {df_monthly.index[-1].strftime('%B %Y')}")
print(f"Missing values: {df_monthly.isnull().sum().sum()}")
print()

summary = df_monthly.describe().T[['mean', 'std', 'min', 'max']]
summary.columns = ['Mean Index', 'Std Dev', 'Min', 'Max']
summary['Range'] = summary['Max'] - summary['Min']
summary = summary.round(2)
print(summary)
print()
print("→ This DataFrame (df_monthly) is the input to all subsequent steps.")
print("→ Next: Normalization, sequence generation, and LSTM training.")